# Prácticas de apoyo · extracción de información

Datos ficticios salvo la consulta real identificada. Conserva los cuadernos originales como referencia. Lee el objetivo, predice, ejecuta y explica el resultado. Reinicia y ejecuta en orden antes de entregar.

Necesitas pandas, requests y beautifulsoup4. Si falta alguna, en una celda de Colab ejecuta `%pip install pandas requests beautifulsoup4`.

## CSV: esperamos solo Ana

In [ ]:
import pandas as pd
from io import StringIO
csv = """nombre,edad,ciudad
Ana,35,Madrid
Luis,30,Madrid
Eva,42,Santander
"""
df = pd.read_csv(StringIO(csv))
mascara = (df["ciudad"] == "Madrid") & (df["edad"] > 30)
resultado = df.loc[mascara, ["nombre", "edad", "ciudad"]]
print(resultado.to_string(index=False))
assert resultado["nombre"].tolist() == ["Ana"]

## JSON simulado: máxima 13 °C en dos horas ficticias

In [ ]:
import pandas as pd
datos = {
    "hourly_units": {"temperature_2m": "°C"},
    "hourly": {"time": ["2024-02-01T10:00", "2024-02-01T11:00"],
               "temperature_2m": [12.5, 13.0]}
}
horas = datos["hourly"]["time"]
valores = datos["hourly"]["temperature_2m"]
assert len(horas) == len(valores) and len(horas) > 0
assert datos["hourly_units"]["temperature_2m"] == "°C"
tabla = pd.DataFrame({"hora_local": pd.to_datetime(horas), "temperatura_c": valores})
assert tabla["temperatura_c"].notna().all()
print(tabla)
print("Máxima:", tabla["temperatura_c"].max())

## HTML: tres productos, uno sin precio

In [ ]:
from bs4 import BeautifulSoup
from decimal import Decimal
import pandas as pd
documento = """<article class="producto"><h2>Cuaderno</h2><span class="precio">3,50 €</span></article>
<article class="producto"><h2>Lápiz</h2><span class="precio">1,20 €</span></article>
<article class="producto"><h2>Carpeta</h2></article>"""
soup = BeautifulSoup(documento, "html.parser")
filas = []
for producto in soup.select("article.producto"):
    nombre = producto.select_one("h2")
    precio = producto.select_one(".precio")
    texto = precio.get_text(strip=True) if precio else None
    importe = Decimal(texto.replace("€", "").strip().replace(",", ".")) if texto else None
    filas.append({"producto": nombre.get_text(strip=True) if nombre else None,
                  "precio_eur": importe, "incidencia": "sin precio" if precio is None else ""})
productos = pd.DataFrame(filas)
print(productos)
assert len(productos) == 3
assert productos["precio_eur"].isna().sum() == 1

## SQL: esperamos Ana, como en pandas

In [ ]:
import sqlite3
import pandas as pd
clientes = pd.DataFrame({"nombre": ["Ana", "Luis", "Eva"],
                         "edad": [35, 30, 42], "ciudad": ["Madrid", "Madrid", "Santander"]})
with sqlite3.connect(":memory:") as conexion:
    clientes.to_sql("clientes", conexion, index=False)
    consulta = "SELECT nombre, edad FROM clientes WHERE ciudad = ? AND edad > ? ORDER BY nombre"
    resultado_sql = pd.read_sql_query(consulta, conexion, params=("Madrid", 30))
print(resultado_sql)
assert resultado_sql["nombre"].tolist() == ["Ana"]

## Calidad: secuencia sobre el mismo conjunto ficticio

Cuatro filas originales; tres después de quitar una copia. Una edad ausente y una fecha inválida en las filas finales. Las fechas con barras usan día/mes/año; las ISO, año-mes-día. Lee la explicación y las reglas en el apartado 3.2 de la web.

In [ ]:
import pandas as pd
from io import StringIO
bruto = pd.read_csv(StringIO("""id,nombre,edad,ciudad,fecha_registro
1,Ana,35,Madird,01/02/2024
2,Luis,, Madrid ,2024-02-02
2,Luis,, Madrid ,2024-02-02
3,Eva,42,Santander,31/02/2024
"""))
print(bruto.shape)
print(bruto.dtypes)
print(bruto.isna().sum())
print("Duplicados completos:", bruto.duplicated().sum())

In [ ]:
limpio = bruto.drop_duplicates().copy()
conflictos = limpio[limpio.duplicated("id", keep=False)]
print("Conflictos por id:", len(conflictos))
limpio["ciudad"] = limpio["ciudad"].str.strip().replace({"Madird": "Madrid"})
print(limpio[["id", "ciudad"]])

In [ ]:
texto = limpio["fecha_registro"].astype("string").str.strip()
fechas = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
iso = texto.str.fullmatch(r"\d{4}-\d{2}-\d{2}", na=False)
fechas.loc[iso] = pd.to_datetime(texto.loc[iso], format="%Y-%m-%d", errors="coerce")
limpio["fecha"] = fechas
incidencias = limpio.loc[texto.notna() & fechas.isna(), ["id", "fecha_registro"]]
print(incidencias)
print(limpio["fecha"].dt.strftime("%d/%m/%Y"))

In [ ]:
assert len(limpio) == 3
assert limpio["id"].is_unique
assert limpio["edad"].dropna().between(0, 120).all()
assert limpio["fecha"].isna().sum() == 1
resumen = limpio.groupby("ciudad", dropna=False).size().rename("registros")
print(resumen)
assert resumen.sum() == len(limpio)

## Consulta real opcional · Santander

Requiere Internet. Es un pronóstico, no una observación. No se ejecuta por defecto para que el resto de las prácticas funcione sin consultas externas. Cambia CONSULTAR_API a True. Conserva parámetros, unidades y fecha. Si falla, explica el error; no presentes datos simulados como actuales.

In [ ]:
CONSULTAR_API = False
if CONSULTAR_API:
    import requests
    from datetime import datetime, timezone
    params = {"latitude":43.44083,"longitude":-3.82238,"hourly":"temperature_2m","forecast_days":1,"timezone":"Europe/Madrid"}
    r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=20)
    r.raise_for_status()
    real = r.json()
    horas = real["hourly"]["time"]
    temperaturas = real["hourly"]["temperature_2m"]
    assert len(horas) == len(temperaturas) and len(horas)>0
    assert real["hourly_units"]["temperature_2m"] == "°C"
    pronostico = pd.DataFrame({"hora_local":pd.to_datetime(horas),"temperatura_c":temperaturas})
    assert pronostico["temperatura_c"].notna().all()
    print("Consulta UTC:",datetime.now(timezone.utc).isoformat(),"Zona:",real["timezone"])
    print(pronostico)
    print(pronostico["temperatura_c"].agg(["min","max"]))
else:
    print("Consulta real desactivada. No se han obtenido datos actuales.")

## Actividad 1

Consulta el enunciado ampliado en la web. Escribe aquí tu predicción, decisiones, controles y conclusión. Identifica si usas la fuente original o los datos ficticios de apoyo.

In [ ]:
# Completa tu solución y explica los resultados.


## Actividad 2

Consulta el enunciado ampliado en la web. Escribe aquí tu predicción, decisiones, controles y conclusión. Identifica si usas la fuente original o los datos ficticios de apoyo.

In [ ]:
# Completa tu solución y explica los resultados.


## Actividad 3

Consulta el enunciado ampliado en la web. Escribe aquí tu predicción, decisiones, controles y conclusión. Identifica si usas la fuente original o los datos ficticios de apoyo.

In [ ]:
# Completa tu solución y explica los resultados.


## Actividad 4

Consulta el enunciado ampliado en la web. Escribe aquí tu predicción, decisiones, controles y conclusión. Identifica si usas la fuente original o los datos ficticios de apoyo.

In [ ]:
# Completa tu solución y explica los resultados.
